In [1]:
import pandas as pd

In [3]:
df = pd.read_excel("../data/Dataset_Procurement.xlsx", sheet_name="Data")
df.head()

,PO Number,PO Date,PO Year,PO Quarter,PO Month,PO Type,PO Status,Supplier ID,Supplier Name,Supplier Country,...,Contract Start,Contract End,Invoice Status,Payment Status,Invoice Match Type,Maverick Spend,Single Source Flag,Preferred Supplier,Local International,Supplier ESG Score
0,PO-00001,09/05/2024,2024,Q2,May,Standard,Closed,SUP-010,SunRise Manufacturing,South Korea,...,17/09/2023,15/09/2025,Disputed,Paid,3-Way Match,No,No,Yes,International,41.6
1,PO-00002,11/01/2024,2024,Q1,January,Standard,Open,SUP-012,Cornerstone Services,United Kingdom,...,17/01/2023,14/06/2025,Overdue,Overdue,2-Way Match,No,No,No,Local,67.8
2,PO-00003,17/02/2024,2024,Q1,February,Standard,Closed,SUP-015,Iron Gate Steel,Poland,...,17/02/2024,18/03/2024,Overdue,Pending,2-Way Match,No,No,No,International,75.7
3,PO-00004,23/06/2023,2023,Q2,June,Blanket,Closed,SUP-007,Nordic Office Solutions,Sweden,...,23/06/2023,23/07/2023,Disputed,Paid,2-Way Match,No,No,No,International,89.1
4,PO-00005,24/01/2022,2022,Q1,January,Emergency,Open,SUP-002,TechPro Components,Germany,...,03/03/2021,31/01/2023,Pending,Overdue,No Match,No,No,Yes,International,41.4


In [4]:
numeric_expected = [
    "PO Year", "Supplier Tier", "Supplier Latitude", "Supplier Longitude",
    "Unit Price", "Quantity", "Discount Pct", "Discount Amount", "Tax Pct",
    "Tax Amount", "Line Total Gross", "Line Net", "Line Total Inc Tax",
    "Budget Unit Price", "Budget Total", "Savings Amount", "Savings Pct",
    "Days Late", "Lead Time Days", "Supplier ESG Score",
]

type_issues = []
for column in numeric_expected:
    if column not in df.columns:
        type_issues.append({"column": column, "issue": "expected column is missing"})
        continue
    if not pd.api.types.is_numeric_dtype(df[column]):
        non_numeric = pd.to_numeric(df[column], errors="coerce").isna() & df[column].notna()
        type_issues.append({
            "column": column,
            "expected": "numeric",
            "actual": str(df[column].dtype),
            "unparseable_values": int(non_numeric.sum()),
            "examples": df.loc[non_numeric, column].head(5).tolist(),
        })

print("Observed column types:")
display(df.dtypes.to_frame(name="dtype"))
print(f"Numeric type issues: {len(type_issues)}")
display(pd.DataFrame(type_issues))

Dataframe column types:


,dtype
PO Number,str
PO Date,str
PO Year,int64
PO Quarter,str
PO Month,str
PO Type,str
PO Status,str
Supplier ID,str
Supplier Name,str
Supplier Country,str


Numeric column ranges:


,min,max
PO Year,2022.0000,2.024000e+03
Supplier Tier,1.0000,3.000000e+00
Supplier Latitude,-25.2744,6.012820e+01
Supplier Longitude,-95.7129,1.382529e+02
Unit Price,0.5400,4.987942e+04
Quantity,1.0000,2.000000e+02
Discount Pct,0.0000,1.500000e+01
Discount Amount,0.0000,7.070166e+05
Tax Pct,0.0000,2.000000e+01
Tax Amount,0.0000,1.525706e+06


In [5]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
print(f"Shape: {df.shape}")

Rows: 5,200
Columns: 57
Shape: (5200, 57)


In [6]:
missing_audit = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": df.isna().mean().mul(100).round(2),
})
display(missing_audit.sort_values("missing_count", ascending=False))

,missing_count,missing_pct
PO Number,0,0.0
PO Date,0,0.0
PO Year,0,0.0
PO Quarter,0,0.0
PO Month,0,0.0
PO Type,0,0.0
PO Status,0,0.0
Supplier ID,0,0.0
Supplier Name,0,0.0
Supplier Country,0,0.0


In [7]:
range_rules = {
    "Supplier Tier": (1, 3),
    "Supplier Latitude": (-90, 90),
    "Supplier Longitude": (-180, 180),
    "Unit Price": (0, None),
    "Quantity": (0, None),
    "Discount Pct": (0, 100),
    "Discount Amount": (0, None),
    "Tax Pct": (0, 100),
    "Tax Amount": (0, None),
    "Line Total Gross": (0, None),
    "Line Net": (0, None),
    "Line Total Inc Tax": (0, None),
    "Budget Unit Price": (0, None),
    "Budget Total": (0, None),
    "Savings Pct": (-100, 100),
    "Lead Time Days": (0, None),
    "Supplier ESG Score": (0, 100),
}

range_issues = []
for column, (lower, upper) in range_rules.items():
    if column not in df.columns:
        range_issues.append({"column": column, "issue": "expected column is missing"})
        continue
    values = pd.to_numeric(df[column], errors="coerce")
    invalid = values.notna()
    if lower is not None:
        invalid &= values.ge(lower) == False
    if upper is not None:
        invalid |= values.notna() & values.gt(upper)
    for index, value in df.loc[invalid, column].items():
        range_issues.append({
            "row_index": index,
            "column": column,
            "value": value,
            "valid_range": f"[{lower if lower is not None else '-inf'}, {upper if upper is not None else 'inf'}]",
        })

print(f"Invalid range values: {len(range_issues)}")
display(pd.DataFrame(range_issues))

Invalid range values: 0


""


In [8]:
duplicate_rows = df.loc[df.duplicated(keep=False)].sort_index()
print(f"Fully duplicated rows (including each copy): {len(duplicate_rows)}")
display(duplicate_rows)

Fully duplicated rows (including each copy): 0


,PO Number,PO Date,PO Year,PO Quarter,PO Month,PO Type,PO Status,Supplier ID,Supplier Name,Supplier Country,...,Contract Start,Contract End,Invoice Status,Payment Status,Invoice Match Type,Maverick Spend,Single Source Flag,Preferred Supplier,Local International,Supplier ESG Score


In [9]:
primary_key = "PO Number"
if primary_key not in df.columns:
    print(f"Cannot check unique identifiers: {primary_key!r} is not a column.")
else:
    key_values = df[primary_key]
    duplicate_keys = key_values.notna() & key_values.duplicated(keep=False)
    print(f"Missing {primary_key} values: {int(key_values.isna().sum())}")
    print(f"Rows with duplicated {primary_key}: {int(duplicate_keys.sum())}")
    display(df.loc[duplicate_keys].sort_values(primary_key))

Missing PO Number values: 0
Rows with duplicated PO Number: 0


,PO Number,PO Date,PO Year,PO Quarter,PO Month,PO Type,PO Status,Supplier ID,Supplier Name,Supplier Country,...,Contract Start,Contract End,Invoice Status,Payment Status,Invoice Match Type,Maverick Spend,Single Source Flag,Preferred Supplier,Local International,Supplier ESG Score


In [10]:
date_columns = [
    "PO Date", "Requested Delivery", "Actual Delivery", "Contract Start", "Contract End",
]
date_issues = []
parsed_dates = {}
for column in date_columns:
    if column not in df.columns:
        date_issues.append({"column": column, "issue": "expected column is missing"})
        continue
    parsed = pd.to_datetime(df[column], format="%d/%m/%Y", errors="coerce")
    parsed_dates[column] = parsed
    invalid = df[column].notna() & parsed.isna()
    for index, value in df.loc[invalid, column].items():
        date_issues.append({"row_index": index, "column": column, "value": value, "issue": "invalid or unparseable date (expected DD/MM/YYYY)"})

if "Contract Start" in parsed_dates and "Contract End" in parsed_dates:
    invalid_order = (
        parsed_dates["Contract Start"].notna()
        & parsed_dates["Contract End"].notna()
        & parsed_dates["Contract End"].lt(parsed_dates["Contract Start"])
    )
    for index in df.index[invalid_order]:
        date_issues.append({
            "row_index": index,
            "column": "Contract Start / Contract End",
            "value": f"{df.at[index, 'Contract Start']} / {df.at[index, 'Contract End']}",
            "issue": "contract end precedes contract start",
        })

print(f"Invalid date or date-order issues: {len(date_issues)}")
display(pd.DataFrame(date_issues))

Invalid date or date-order issues: 0


""
